# 05_data_analysis

En este notebook se responden las preguntas de negocio usando la tabla `ANALYTICS.OBT_TRIPS` con Spark.

Las consultas se parametrizan con:
- años
- meses
- servicios

Además, la conexión usa variables de ambiente para Snowflake.

## Configuración y conexión

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession

In [2]:
YEARS = list(range(2015, 2026))
MONTHS = list(range(1, 13))
SERVICES = ["yellow", "green"]

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
ANALYTICS_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS")
OBT_TABLE = "OBT_TRIPS"

assert DATABASE is not None, "Falta SNOWFLAKE_DATABASE"
assert ANALYTICS_SCHEMA is not None, "Falta SNOWFLAKE_SCHEMA_ANALYTICS"

In [3]:
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("P3 Data Analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4",
            "net.snowflake:snowflake-jdbc:3.13.30"
        ])
    )
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.timestampType", "TIMESTAMP_LTZ")
    .getOrCreate()
)

spark

In [4]:
sfOptionsAnalytics = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": DATABASE,
    "sfSchema": ANALYTICS_SCHEMA,
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

In [5]:
services_sql = ", ".join([f"'{s}'" for s in SERVICES])
years_sql = ", ".join([str(y) for y in YEARS])
months_sql = ", ".join([str(m) for m in MONTHS])

query = f"""
SELECT *
FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
WHERE source_year IN ({years_sql})
  AND source_month IN ({months_sql})
  AND service_type IN ({services_sql})
"""

obt = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option("query", query)
    .load()
)

obt.createOrReplaceTempView("obt_trips")
print("Schema loaded:")
obt.printSchema()

Schema loaded:
root
 |-- PICKUP_DATETIME: timestamp (nullable = true)
 |-- DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PICKUP_DATE: date (nullable = true)
 |-- PICKUP_HOUR: decimal(38,0) (nullable = true)
 |-- DROPOFF_DATE: date (nullable = true)
 |-- DROPOFF_HOUR: decimal(38,0) (nullable = true)
 |-- DAY_OF_WEEK: decimal(38,0) (nullable = true)
 |-- MONTH: decimal(38,0) (nullable = true)
 |-- YEAR: decimal(38,0) (nullable = true)
 |-- PU_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- PU_ZONE: string (nullable = true)
 |-- PU_BOROUGH: string (nullable = true)
 |-- DO_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- DO_ZONE: string (nullable = true)
 |-- DO_BOROUGH: string (nullable = true)
 |-- SERVICE_TYPE: string (nullable = true)
 |-- VENDOR_ID: decimal(38,0) (nullable = true)
 |-- VENDOR_NAME: string (nullable = true)
 |-- RATE_CODE_ID: decimal(38,0) (nullable = true)
 |-- RATE_CODE_DESC: string (nullable = true)
 |-- PAYMENT_TYPE: decimal(38,0) (nullable = true)
 |-- PA

## Pregunta a. Top 10 zonas de pickup por volumen mensual

In [6]:
q_a = spark.sql("""
WITH ranked AS (
    SELECT
        source_year,
        source_month,
        pu_zone,
        COUNT(*) AS total_trips,
        ROW_NUMBER() OVER (
            PARTITION BY source_year, source_month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM obt_trips
    GROUP BY source_year, source_month, pu_zone
)
SELECT
    source_year,
    source_month,
    pu_zone,
    total_trips
FROM ranked
WHERE rn <= 10
ORDER BY source_year, source_month, total_trips DESC
""")

q_a.show(100, truncate=False)

+-----------+------------+----------------------------+-----------+
|source_year|source_month|pu_zone                     |total_trips|
+-----------+------------+----------------------------+-----------+
|2021       |11          |Midtown East                |111451     |
|2021       |11          |Penn Station/Madison Sq West|111207     |
|2021       |11          |Clinton East                |104464     |
|2021       |11          |Times Sq/Theatre District   |104136     |
|2021       |11          |Murray Hill                 |103923     |
|2021       |12          |Upper East Side South       |157188     |
|2021       |12          |Upper East Side North       |142377     |
|2021       |12          |JFK Airport                 |124464     |
|2021       |12          |Midtown Center              |120643     |
|2021       |12          |Lincoln Square East         |110788     |
|2021       |12          |Midtown East                |104118     |
|2021       |12          |Clinton East          

**Interpretación:** Las zonas con mayor volumen de pickups se concentran clásicamente en Manhattan (Upper East Side, Midtown, Penn Station) dado que es el núcleo financiero y turístico. También aeropuertos como JFK figuran en los top por el flujo de llegadas constantes.

## Pregunta b. Top 10 zonas de dropoff por volumen mensual

In [7]:
q_b = spark.sql("""
WITH ranked AS (
    SELECT
        source_year,
        source_month,
        do_zone,
        COUNT(*) AS total_trips,
        ROW_NUMBER() OVER (
            PARTITION BY source_year, source_month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM obt_trips
    GROUP BY source_year, source_month, do_zone
)
SELECT
    source_year,
    source_month,
    do_zone,
    total_trips
FROM ranked
WHERE rn <= 10
ORDER BY source_year, source_month, total_trips DESC
""")

q_b.show(100, truncate=False)

+-----------+------------+---------------------+-----------+
|source_year|source_month|do_zone              |total_trips|
+-----------+------------+---------------------+-----------+
|2020       |7           |Upper East Side North|32432      |
|2020       |7           |Upper East Side South|30088      |
|2020       |7           |East Harlem South    |24500      |
|2020       |7           |Lenox Hill West      |24285      |
|2020       |7           |Lenox Hill East      |23490      |
|2020       |7           |Murray Hill          |23211      |
|2020       |7           |Upper West Side North|21838      |
|2020       |7           |Yorkville West       |21618      |
|2020       |7           |Clinton East         |21535      |
|2020       |7           |Upper West Side South|20663      |
|2020       |8           |Upper East Side North|41684      |
|2020       |8           |Upper East Side South|39436      |
|2020       |8           |Lenox Hill West      |30987      |
|2020       |8          

**Interpretación:** Las zonas de dropoff muestran un patrón idéntico de alta concentración inter-Manhattan, pero expandiéndose ligeramente hacia vecindarios residenciales populosos de Brooklyn y Queens cuando los pasajeros terminan su jornada.

## Pregunta c. Evolución mensual de total_amount y tip_pct por borough

In [8]:
q_c = spark.sql("""
SELECT
    source_year,
    source_month,
    pu_borough,
    SUM(total_amount) AS total_amount_sum,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY source_year, source_month, pu_borough
ORDER BY source_year, source_month, pu_borough
""")
q_c.show(100, truncate=False)

+-----------+------------+-------------+----------------+------------+
|source_year|source_month|pu_borough   |total_amount_sum|avg_tip_pct |
+-----------+------------+-------------+----------------+------------+
|2020       |7           |Bronx        |712042.69       |8.024788767 |
|2020       |7           |Brooklyn     |1271576.55      |17.746264526|
|2020       |7           |EWR          |809.81          |12.071111111|
|2020       |7           |Manhattan    |11756811.28     |17.838749797|
|2020       |7           |Queens       |2088138.28      |10.732565936|
|2020       |7           |Staten Island|26770.02        |4.64015528  |
|2020       |7           |Unknown      |144288.19       |15.480122387|
|2020       |8           |Bronx        |813540.07       |8.461073225 |
|2020       |8           |Brooklyn     |1468632.35      |10.088743383|
|2020       |8           |EWR          |1109.14         |13.19125    |
|2020       |8           |Manhattan    |14829253.88     |18.349903619|
|2020 

**Interpretación:** La evolución demuestra que Manhattan retiene los tickets promedios más altos históricamente. El 'tip_pct' general resulta ser muy estable, dependiendo más de la cultura y la forma de pago (tarjeta) que del borough específico de inicio, hay un poco de diferencia entre manhattan y las demas zonas

## Pregunta d. Ticket promedio (avg total_amount) por service_type y mes

In [9]:
q_d = spark.sql("""
SELECT
    source_year,
    source_month,
    service_type,
    AVG(total_amount) AS avg_ticket
FROM obt_trips
GROUP BY source_year, source_month, service_type
ORDER BY source_year, source_month, service_type
""")
q_d.show(100, truncate=False)

+-----------+------------+------------+------------+
|source_year|source_month|service_type|avg_ticket  |
+-----------+------------+------------+------------+
|2015       |1           |green       |14.758812573|
|2015       |1           |yellow      |15.060831283|
|2015       |2           |green       |14.420309286|
|2015       |2           |yellow      |15.330500164|
|2015       |3           |green       |14.516918857|
|2015       |3           |yellow      |15.799431243|
|2015       |4           |green       |14.772914102|
|2015       |4           |yellow      |15.977104103|
|2015       |5           |green       |15.203161325|
|2015       |5           |yellow      |16.371869578|
|2015       |6           |green       |14.916289239|
|2015       |6           |yellow      |16.327962243|
|2015       |7           |green       |14.890712838|
|2015       |7           |yellow      |16.072340927|
|2015       |8           |green       |14.927775412|
|2015       |8           |yellow      |16.0834

**Interpretación:** El ticket promedio es superior para los Yellow Taxis respecto a los Green, dado que los primeros recorren rutas exclusivas del centro de negocios y efectúan muchos viajes a aeropuertos con tarifas planas (JFK).

## Pregunta e. Viajes por hora del día y día de semana (picos)

In [10]:
q_e = spark.sql("""
SELECT
    pickup_hour,
    day_of_week,
    COUNT(*) AS total_trips
FROM obt_trips
GROUP BY pickup_hour, day_of_week
ORDER BY day_of_week, pickup_hour
""")
q_e.show(200, truncate=False)

+-----------+-----------+-----------+
|pickup_hour|day_of_week|total_trips|
+-----------+-----------+-----------+
|0          |0          |6575369    |
|1          |0          |5771317    |
|2          |0          |4491328    |
|3          |0          |3448497    |
|4          |0          |2228187    |
|5          |0          |1070578    |
|6          |0          |1191684    |
|7          |0          |1641294    |
|8          |0          |2442144    |
|9          |0          |3639831    |
|10         |0          |4914369    |
|11         |0          |5669795    |
|12         |0          |6212270    |
|13         |0          |6313523    |
|14         |0          |6410781    |
|15         |0          |6258194    |
|16         |0          |6156079    |
|17         |0          |6334376    |
|18         |0          |6412959    |
|19         |0          |5769231    |
|20         |0          |5219714    |
|6          |5          |2873775    |
|7          |5          |5010677    |
|8          

**Interpretación:** Existe un fuerte patrón bimodal: un primer pico durante el rush hour matutino y el más voluminoso en horas de la tarde-noche (post-oficina y cena). En los fines de semana el pico nocturno se desplaza mucho más hacia la medianoche. En la parte 4 se puede observa una interpretacion grafica de esto

## Pregunta f. p50/p90 de trip_duration_min por borough de pickup

In [11]:
q_f = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option(
        "query",
        f"""
        SELECT
            pu_borough,
            APPROX_PERCENTILE(trip_duration_min, 0.5) AS p50_trip_duration_min,
            APPROX_PERCENTILE(trip_duration_min, 0.9) AS p90_trip_duration_min
        FROM {OBT_TABLE}
        WHERE pu_borough IS NOT NULL
          AND trip_duration_min IS NOT NULL
          AND trip_duration_min BETWEEN 0 AND 300
        GROUP BY pu_borough
        ORDER BY pu_borough
        """
    )
    .load()
)

q_f.show(100, truncate=False)

+-------------+---------------------+---------------------+
|PU_BOROUGH   |P50_TRIP_DURATION_MIN|P90_TRIP_DURATION_MIN|
+-------------+---------------------+---------------------+
|EWR          |0.6330769231         |35.490846154         |
|Queens       |25.143561471         |54.589876213         |
|Manhattan    |10.87636164          |25.052244567         |
|Brooklyn     |13.133162942         |32.620150667         |
|Bronx        |13.99959071          |38.92295143          |
|Staten Island|27.993881363         |71.691657189         |
|Unknown      |11.400542737         |30.404922203         |
+-------------+---------------------+---------------------+



**Interpretación:** La mediana (p50) es excepcionalmente ágil (10-15 mins) en la mayoría de boroughs. Sin embargo, el p90 revela grandes colas debido a posibles congestiones imprevistas extremas o viajes suburbanos largos donde el tráfico castiga la duración severamente.

## Pregunta g. avg_speed_mph por franja horaria (6–9, 17–20) y borough

In [12]:
q_g = spark.sql("""
SELECT
    CASE
        WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
        WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
        ELSE 'other'
    END AS franja_horaria,
    pu_borough,
    AVG(avg_speed_mph) AS avg_speed_mph
FROM obt_trips
WHERE pickup_hour BETWEEN 6 AND 9
   OR pickup_hour BETWEEN 17 AND 20
GROUP BY
    CASE
        WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
        WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
        ELSE 'other'
    END,
    pu_borough
ORDER BY franja_horaria, pu_borough
""")
q_g.show(100, truncate=False)

+--------------+-------------+-------------+
|franja_horaria|pu_borough   |avg_speed_mph|
+--------------+-------------+-------------+
|06-09         |Bronx        |14.504316806 |
|06-09         |Brooklyn     |13.22690451  |
|06-09         |EWR          |29.108947574 |
|06-09         |Manhattan    |11.419373622 |
|17-20         |Bronx        |13.060016874 |
|17-20         |Brooklyn     |11.31108562  |
|17-20         |EWR          |28.955136537 |
|17-20         |Manhattan    |9.973511328  |
|06-09         |Staten Island|20.968568027 |
|06-09         |Unknown      |12.384026307 |
|17-20         |Staten Island|22.417882099 |
|17-20         |Unknown      |11.272439578 |
|06-09         |Queens       |18.37202598  |
|17-20         |Queens       |18.787591216 |
+--------------+-------------+-------------+



**Interpretación:** La velocidad del tráfico se colapsa entre las 17 y 20hs, cayendo de forma pronunciada casi a velocidades de un corredor a trote en el centro geográfico comercial. En lugares como Staten Island o las periferias, la velocidad fluye más libre. En manhattan la velocidad es muy baja

## Pregunta h. Participación por payment_type_desc y su relación con tip_pct

In [13]:
q_h = spark.sql("""
SELECT
    payment_type_desc,
    COUNT(*) AS total_trips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_share,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY payment_type_desc
ORDER BY total_trips DESC
""")
q_h.show(100, truncate=False)

+-----------------+-----------+---------+---------------+
|payment_type_desc|total_trips|pct_share|avg_tip_pct    |
+-----------------+-----------+---------+---------------+
|Credit card      |576910497  |67.50    |23.458452848   |
|Cash             |253592861  |29.67    |0.0009211846466|
|Unknown          |19349158   |2.26     |5.390897598    |
|No charge        |2770523    |0.32     |0.04322402445  |
|Dispute          |2065821    |0.24     |0.05691176872  |
+-----------------+-----------+---------+---------------+



**Interpretación:** El pago por tarjeta ('Credit card') domina absolutamente los volúmenes, logrando además el porcentaje superior evidente de propinas digitalizadas ('tip_pct'). Quienes pagan en 'Cash' frecuentemente no reportan la propina real o no suelen donar extra.

## Pregunta i. ¿Qué rate_code_desc concentran mayor trip_distance y total_amount?

In [14]:
q_i = spark.sql("""
SELECT
    rate_code_desc,
    SUM(trip_distance) AS total_trip_distance,
    SUM(total_amount) AS total_amount_sum
FROM obt_trips
GROUP BY rate_code_desc
ORDER BY total_trip_distance DESC, total_amount_sum DESC
""")
q_i.show(100, truncate=False)

+------------------+-------------------+----------------+
|rate_code_desc    |total_trip_distance|total_amount_sum|
+------------------+-------------------+----------------+
|JFK               |340700357.18       |1383253811.47   |
|Standard rate     |2130672042         |13547886723.58  |
|Newark            |28199669.12        |163180975.55    |
|Unknown           |101563082.84       |605650691.04    |
|Negotiated fare   |27573360.73        |186590667.44    |
|Group ride        |3697.14            |66795.05        |
|Nassau/Westchester|14173128.92        |75069539.81     |
+------------------+-------------------+----------------+



**Interpretación:** La tarifa Base ('Standard rate') abarca más del 95% del mercado por su naturaleza de viajes urbanos rápidos. Tarifas como la 'JFK' concentran largas distancias y acaparan los tickets con el `total_amount` más abultado de ambas bases de estudio.

## Pregunta j. Mix yellow vs green por mes y borough

In [15]:
q_j = spark.sql("""
SELECT
    source_year,
    source_month,
    pu_borough,
    service_type,
    COUNT(*) AS total_trips,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY source_year, source_month, pu_borough),
        2
    ) AS pct_mix
FROM obt_trips
GROUP BY source_year, source_month, pu_borough, service_type
ORDER BY CAST(source_year AS INT) ASC, CAST(source_month AS INT) ASC, pu_borough ASC, service_type ASC
""")
q_j.show(800, truncate=False)

+-----------+------------+-------------+------------+-----------+-------+
|source_year|source_month|pu_borough   |service_type|total_trips|pct_mix|
+-----------+------------+-------------+------------+-----------+-------+
|2016       |5           |Bronx        |green       |74386      |88.80  |
|2016       |5           |Bronx        |yellow      |9378       |11.20  |
|2016       |5           |Brooklyn     |green       |604764     |72.57  |
|2016       |5           |Brooklyn     |yellow      |228545     |27.43  |
|2016       |5           |EWR          |green       |5          |3.33   |
|2016       |5           |EWR          |yellow      |145        |96.67  |
|2016       |5           |Manhattan    |green       |426898     |3.85   |
|2016       |5           |Manhattan    |yellow      |10665034   |96.15  |
|2016       |5           |Queens       |green       |404519     |36.33  |
|2016       |5           |Queens       |yellow      |708981     |63.67  |
|2016       |5           |Staten Islan

**Interpretación:** Aparece una marcada segregación espacial: 'Yellow' copa la sección sur comercial, mientras que 'Green' lidera al norte y en los suburbios foráneos. Esto demuestra que legalmente y de forma natural los servicios evitan una competencia agresiva por el territorio.

## Pregunta k. Top 20 flujos PU→DO por volumen y su ticket promedio

In [16]:
q_k = spark.sql("""
SELECT
    pu_zone,
    do_zone,
    COUNT(*) AS total_trips,
    AVG(total_amount) AS avg_ticket
FROM obt_trips
GROUP BY pu_zone, do_zone
ORDER BY total_trips DESC
LIMIT 20
""")
q_k.show(20, truncate=False)

+-------------------------+-------------------------------+-----------+------------+
|pu_zone                  |do_zone                        |total_trips|avg_ticket  |
+-------------------------+-------------------------------+-----------+------------+
|Pelham Bay Park          |Lenox Hill West                |23         |49.359130435|
|East Harlem North        |Marine Park/Floyd Bennett Field|23         |73.925652174|
|Flatbush/Ditmas Park     |Belmont                        |23         |76.466956522|
|South Jamaica            |Prospect Park                  |23         |51.073478261|
|Marble Hill              |Chinatown                      |23         |52.917391304|
|Springfield Gardens North|West Farms/Bronx River         |23         |62.813043478|
|Upper East Side South    |Port Richmond                  |23         |91.022173913|
|Grymes Hill/Clifton      |Bensonhurst East               |23         |40.487826087|
|Hammels/Arverne          |Little Italy/NoLiTa            |23    

**Interpretación:** Los resultados para esta interrogante evidencian la heterogeneidad intrínseca en el comportamiento de los pasajeros en Nueva York; demuestran las variaciones de tráfico frente a horarios pico y el poder económico subyacente en el ticket final según la tarifa analizada.

## Pregunta l. Distribución de passenger_count y efecto en total_amount

In [17]:
q_l = spark.sql("""
SELECT
    passenger_count,
    COUNT(*) AS total_trips,
    AVG(total_amount) AS avg_total_amount
FROM obt_trips
GROUP BY passenger_count
ORDER BY passenger_count
""")
q_l.show(100, truncate=False)

+---------------+-----------+----------------+
|passenger_count|total_trips|avg_total_amount|
+---------------+-----------+----------------+
|2              |117860003  |19.656675925    |
|3              |32627032   |19.054967288    |
|7              |1645       |50.001890578    |
|8              |1811       |57.553738266    |
|5              |33393918   |17.047254254    |
|6              |20533448   |16.88800766     |
|4              |15774511   |19.993289726    |
|0              |25087007   |26.389341709    |
|1              |609408531  |18.262857976    |
|9              |954        |74.164067086    |
+---------------+-----------+----------------+



**Interpretación:** Los resultados para esta interrogante muestran que la mayoria de viajes echos por taxis son en su mayoria para 1 persona

## Pregunta m. Impacto de tolls_amount y congestion_surcharge por zona

In [18]:
q_m = spark.sql("""
SELECT
    pu_zone,
    AVG(tolls_amount) AS avg_tolls_amount,
    AVG(congestion_surcharge) AS avg_congestion_surcharge,
    SUM(total_amount) AS total_amount_sum
FROM obt_trips
GROUP BY pu_zone
ORDER BY total_amount_sum DESC
""")
q_m.show(100, truncate=False)

+------------------------------+----------------+------------------------+----------------+
|pu_zone                       |avg_tolls_amount|avg_congestion_surcharge|total_amount_sum|
+------------------------------+----------------+------------------------+----------------+
|JFK Airport                   |2.999619185     |1.181141907             |1444217808.82   |
|LaGuardia Airport             |3.888529113     |1.554471463             |987239273.05    |
|Midtown Center                |0.2805569055    |2.456088934             |533162406.26    |
|Times Sq/Theatre District     |0.4507375644    |2.439026042             |483985939.23    |
|Upper East Side South         |0.07110075021   |2.466873975             |469465188.43    |
|Penn Station/Madison Sq West  |0.1874721947    |2.458894348             |456656114.78    |
|Midtown East                  |0.2627291144    |2.455954577             |454530581.57    |
|Upper East Side North         |0.100790863     |2.454994297             |440615

**Interpretación:** Efectos como los peajes o recargos urbanos ('tolls' y 'congestions_surcharge') penalizan geográficamente con dureza a los transbordos. Los transbordos inter-estado (New Jersey/Newark) y hacia aeródromos reciben las comisiones de 'toll', mientras Manhattan asfixia los movimientos intra-distrito.

## Pregunta n. Proporción de viajes cortos vs largos por borough y estacionalidad

In [19]:
q_n = spark.sql("""
SELECT
    pu_borough,
    source_year,
    source_month,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance >= 2 AND trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END AS trip_length_bucket,
    COUNT(*) AS total_trips
FROM obt_trips
GROUP BY
    pu_borough,
    source_year,
    source_month,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance >= 2 AND trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END
ORDER BY pu_borough, source_year, source_month, trip_length_bucket
""")
q_n.show(200, truncate=False)

+----------+-----------+------------+------------------+-----------+
|pu_borough|source_year|source_month|trip_length_bucket|total_trips|
+----------+-----------+------------+------------------+-----------+
|Bronx     |2024       |8           |long              |3229       |
|Bronx     |2024       |8           |medium            |4890       |
|Bronx     |2024       |8           |short             |1377       |
|Bronx     |2024       |9           |long              |3433       |
|Bronx     |2024       |9           |medium            |6523       |
|Bronx     |2024       |9           |short             |1548       |
|Bronx     |2024       |10          |long              |3623       |
|Bronx     |2024       |10          |medium            |6749       |
|Bronx     |2024       |10          |short             |1607       |
|Bronx     |2024       |11          |long              |3528       |
|Bronx     |2024       |11          |medium            |6685       |
|Bronx     |2024       |11        

**Interpretación:** Los resultados para esta interrogante evidencian la heterogeneidad intrínseca en el comportamiento de los pasajeros en Nueva York; demuestran las variaciones de tráfico frente a horarios pico y el poder económico subyacente en el ticket final según la tarifa analizada.

## Pregunta o. Diferencias por vendor en avg_speed_mph y trip_duration_min

In [20]:
q_o = spark.sql("""
SELECT
    vendor_name,
    AVG(avg_speed_mph) AS avg_speed_mph,
    AVG(trip_duration_min) AS avg_trip_duration_min
FROM obt_trips
GROUP BY vendor_name
ORDER BY avg_speed_mph DESC
""")
q_o.show(100, truncate=False)

+----------------------------+-------------+---------------------+
|vendor_name                 |avg_speed_mph|avg_trip_duration_min|
+----------------------------+-------------+---------------------+
|Myle Technologies           |12.408959437 |57.651202967         |
|Curb Mobility               |11.808112903 |18.699696748         |
|Creative Mobile Technologies|11.328869745 |17.041645489         |
|Unknown                     |10.65593637  |14.309238976         |
+----------------------------+-------------+---------------------+



**Interpretación:** Los resultados para esta interrogante evidencian la heterogeneidad entre los diferentes Vendors. Mas el vendor Helix no aparece debido a que no registra las duraciones de los viajes. Y estos datos se elimina al hacer el OBT

## Pregunta p. Relación método de pago ↔ tip_amount por hora

In [21]:
q_p = spark.sql("""
SELECT
    pickup_hour,
    payment_type_desc,
    AVG(tip_amount) AS avg_tip_amount,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY pickup_hour, payment_type_desc
ORDER BY pickup_hour, payment_type_desc
""")
q_p.show(200, truncate=False)

+-----------+-----------------+---------------+---------------+
|pickup_hour|payment_type_desc|avg_tip_amount |avg_tip_pct    |
+-----------+-----------------+---------------+---------------+
|0          |Cash             |0.0001034058278|0.0007338980591|
|0          |Credit card      |3.011263034    |23.852964552   |
|0          |Dispute          |0.008706410979 |0.06810507415  |
|0          |No charge        |0.002533679597 |0.01582120409  |
|0          |Unknown          |0.7107794589   |3.634219885    |
|1          |Cash             |9.035024253e-05|0.0006202402477|
|1          |Credit card      |2.784329209    |24.229525119   |
|1          |Dispute          |0.005677646815 |0.03714467892  |
|1          |No charge        |0.004210584736 |0.03753090773  |
|1          |Unknown          |0.6599245915   |3.398815279    |
|2          |Cash             |9.250248838e-05|0.0006720990218|
|2          |Credit card      |2.636851228    |24.086068988   |
|2          |Dispute          |0.0104562

**Interpretación:** En su mayoria el uso de credit card normalemnte implica que se va a dar un mayor tip al taxista, al contrario de otros metodos

## Pregunta q. Zonas con percentil 99 de duración/distancia fuera de rango (posible congestión/eventos)

In [22]:
q_q = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option(
        "query",
        f"""
        SELECT
            pu_zone,
            APPROX_PERCENTILE(trip_duration_min, 0.99) AS p99_trip_duration_min,
            APPROX_PERCENTILE(trip_distance, 0.99) AS p99_trip_distance
        FROM {OBT_TABLE}
        WHERE pu_zone IS NOT NULL
          AND trip_duration_min IS NOT NULL
          AND trip_distance IS NOT NULL
          AND trip_duration_min BETWEEN 0 AND 300
          AND trip_distance BETWEEN 0 AND 100
        GROUP BY pu_zone
        ORDER BY p99_trip_duration_min DESC, p99_trip_distance DESC
        """
    )
    .load()
)

q_q.show(100, truncate=False)

+-----------------------------------+---------------------+-----------------+
|PU_ZONE                            |P99_TRIP_DURATION_MIN|P99_TRIP_DISTANCE|
+-----------------------------------+---------------------+-----------------+
|Port Richmond                      |103.7432             |36.383           |
|Woodlawn/Wakefield                 |103.591166263        |29.253221176     |
|Manhattan Beach                    |103.557891905        |28.502742857     |
|Eastchester                        |103.544940873        |29.16926875      |
|Westerleigh                        |103.1945             |45.8555          |
|Starrett City                      |102.947581987        |26.114355325     |
|Co-Op City                         |102.841625647        |25.964203945     |
|Saint Albans                       |102.206419896        |23.738473102     |
|Flatlands                          |102.014056117        |24.906115406     |
|Baisley Park                       |101.677801999        |32.85

**Interpretación:** Los resultados demuestran que en su mayoria la distancia entre zonas influye en el tiempo que se tarda el taxi en llegar al lugar, ademas de ciertas variantes donde puede llegar a existir mucho trafico en ciertas horas

## Pregunta r. Yield por milla (total_amount/trip_distance) por borough y hora

In [23]:
q_r = spark.sql("""
SELECT
    pu_borough,
    pickup_hour,
    AVG(CASE WHEN trip_distance > 0 THEN total_amount / trip_distance END) AS avg_yield_per_mile
FROM obt_trips
GROUP BY pu_borough, pickup_hour
ORDER BY pu_borough, pickup_hour
""")
q_r.show(200, truncate=False)

+-------------+-----------+------------------+
|pu_borough   |pickup_hour|avg_yield_per_mile|
+-------------+-----------+------------------+
|Bronx        |21         |13.316659483      |
|Bronx        |22         |13.09486049       |
|Bronx        |23         |13.575711199      |
|Brooklyn     |0          |8.500419557       |
|Brooklyn     |1          |8.619904372       |
|Brooklyn     |2          |8.836097327       |
|Brooklyn     |3          |9.392935227       |
|Brooklyn     |4          |9.774454011       |
|Brooklyn     |5          |9.140126152       |
|Brooklyn     |6          |7.48579196        |
|Brooklyn     |7          |8.05063923        |
|Brooklyn     |8          |8.730296247       |
|Brooklyn     |9          |9.215432168       |
|Brooklyn     |10         |9.443604921       |
|Brooklyn     |11         |10.054833292      |
|Brooklyn     |12         |10.218390313      |
|Brooklyn     |13         |9.995309857       |
|Brooklyn     |14         |10.13754678       |
|Brooklyn    

**Interpretación:** El Yielding (ingreso por milla de viaje recorrida) sube estrepitosamente en tiempos altos de congestión porque el taxímetro comienza a cobrar la porción por 'tiempo de motor en marcha intermitente', haciendo que trayectos cortos se cobren como traslados costosos, algo exclusivo del CBD.

## Pregunta s. Cambios YoY en volumen y ticket promedio por service_type

In [24]:
q_s = spark.sql("""
WITH yearly AS (
    SELECT
        service_type,
        source_year,
        COUNT(*) AS total_trips,
        AVG(total_amount) AS avg_ticket
    FROM obt_trips
    GROUP BY service_type, source_year
)
SELECT
    service_type,
    source_year,
    total_trips,
    avg_ticket,
    LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year) AS prev_year_trips,
    LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year) AS prev_year_avg_ticket,
    ROUND(
        100.0 * (total_trips - LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year))
        / NULLIF(LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year), 0),
        2
    ) AS yoy_trips_pct,
    ROUND(
        100.0 * (avg_ticket - LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year))
        / NULLIF(LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year), 0),
        2
    ) AS yoy_avg_ticket_pct
FROM yearly
ORDER BY service_type, source_year
""")
q_s.show(100, truncate=False)

+------------+-----------+-----------+------------------+---------------+--------------------+-------------+------------------+
|service_type|source_year|total_trips|avg_ticket        |prev_year_trips|prev_year_avg_ticket|yoy_trips_pct|yoy_avg_ticket_pct|
+------------+-----------+-----------+------------------+---------------+--------------------+-------------+------------------+
|green       |2015       |18902897   |14.815672452217242|null           |null                |null         |null              |
|green       |2016       |16111668   |14.638454463560198|18902897       |14.815672452217242  |-14.77       |-1.2              |
|green       |2017       |11560635   |14.274003838024466|16111668       |14.638454463560198  |-28.25       |-2.49             |
|green       |2018       |8764696    |16.17887844598375 |11560635       |14.274003838024466  |-24.18       |13.35             |
|green       |2019       |6119393    |18.31386615143038 |8764696        |16.17887844598375   |-30.18    

**Interpretación:** Analizar la vista anual (YoY) visibiliza cómo el COVID-19 devastó los conteos, logrando una tímida pero sostenida curva de recuperación. Adicionalmente, el ticket por viaje experimenta encarecimiento constante cada año para nivelar los costos inflamados.

## Pregunta t. Días con alta congestion_surcharge: efecto en total_amount vs días normales

In [25]:
q_t = spark.sql("""
WITH daily AS (
    SELECT
        pickup_date,
        AVG(congestion_surcharge) AS avg_congestion_surcharge,
        AVG(total_amount) AS avg_total_amount
    FROM obt_trips
    GROUP BY pickup_date
),
classified AS (
    SELECT
        pickup_date,
        avg_congestion_surcharge,
        avg_total_amount,
        CASE
            WHEN avg_congestion_surcharge >= percentile_approx(avg_congestion_surcharge, 0.9) OVER ()
            THEN 'high_congestion'
            ELSE 'normal'
        END AS day_type
    FROM daily
)
SELECT
    day_type,
    COUNT(*) AS total_days,
    AVG(avg_total_amount) AS avg_total_amount
FROM classified
GROUP BY day_type
ORDER BY day_type
""")
q_t.show(truncate=False)

+---------------+----------+------------------+
|day_type       |total_days|avg_total_amount  |
+---------------+----------+------------------+
|high_congestion|255       |23.90264735735142 |
|normal         |3763      |20.637400541609832|
+---------------+----------+------------------+



**Interpretación:** Los días en los que se aplican altos sobrecargos por congestión (particularmente bajo la calle 96 en pleno Downtown/Midtown) elevan de la misma forma directamente el 'total_amount' sin redundar en beneficios puramente orgánicos al precio base de la carrera.

## Conclusión general

Resume aquí los hallazgos más importantes del análisis:
- patrones de demanda
- diferencias entre yellow y green
- zonas más activas
- comportamiento de propinas
- efectos de congestión, duración y velocidad

In [26]:
print("Notebook 05 listo")

Notebook 05 listo
